# Module 4: Human-in-the-Loop (HITL)

**Day 4 — Agents, LangGraph & MCP**

## What you will learn
- Why HITL matters for production agents
- `interrupt()` + `Command(resume=...)` pattern
- Structured action proposals with Pydantic
- `ApprovalWorkflow` pattern

## 1. The Core Idea: Stop and Ask

Before irreversible actions (send email, delete data, deploy), the agent pauses and waits for human approval.

In [ ]:
# Simplest HITL — pure Python
def execute_with_approval(action: str, approved: bool) -> str:
    print(f"  Proposed: {action}")
    print(f"  Decision: {'APPROVED' if approved else 'REJECTED'}")
    if approved:
        return f"✅ Executed: {action}"
    return f"❌ Rejected: {action}"

print(execute_with_approval("Send newsletter to 50,000 users",    approved=True))
print()
print(execute_with_approval("Drop production database table",      approved=False))
print()
print(execute_with_approval("Generate monthly sales report",       approved=True))

## 2. Structured Action Proposals

Pydantic enforces a schema for what the agent proposes. Makes review easier and provides an audit trail.

In [ ]:
from pydantic import BaseModel
from typing import Literal

class ActionProposal(BaseModel):
    action_type: Literal["email", "database", "api_call", "file_write", "deploy"]
    target: str
    description: str
    risk_level: Literal["low", "medium", "high"]
    reversible: bool

proposal = ActionProposal(
    action_type="email",
    target="premium_users@company.com",
    description="Send weekly digest with personalised recommendations",
    risk_level="low",
    reversible=False
)
print(proposal.model_dump_json(indent=2))

# Auto-approve policy
def should_auto_approve(p: ActionProposal) -> bool:
    return p.risk_level == "low"

print(f"\nAuto-approve: {should_auto_approve(proposal)}")

## 3. LangGraph `interrupt()` — How it Works

```
Graph runs: draft_node → [PAUSE at interrupt()] → resume_node
                                ↑
                    Human sees state, calls Command(resume=True/False)
```

The state is saved in the checkpointer. The human can take minutes or hours to respond.

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from typing import TypedDict, Optional

class ApprState(TypedDict):
    request: str
    draft: Optional[str]
    approved: Optional[bool]
    result: Optional[str]

def draft_node(s):   return {"draft": f"Proposed action for: {s['request']}"}
def review_node(s):  decision = interrupt({"draft": s["draft"]}); return {"approved": decision}
def execute_node(s): return {"result": "✅ Done" if s["approved"] else "❌ Rejected"}

g = StateGraph(ApprState)
for name, fn in [("draft", draft_node), ("review", review_node), ("execute", execute_node)]:
    g.add_node(name, fn)
g.set_entry_point("draft")
g.add_edge("draft",   "review")
g.add_edge("review",  "execute")
g.add_edge("execute", END)
app = g.compile(checkpointer=MemorySaver(), interrupt_before=["review"])

cfg = {"configurable": {"thread_id": "t1"}}

# Step 1: Run until interrupt
state = app.invoke({"request": "deploy to production", "draft": None, "approved": None, "result": None}, config=cfg)
print("Paused. State:", {k: v for k, v in state.items() if v is not None})

# Step 2: Human approves
final = app.invoke(Command(resume=True), config=cfg)
print("Result:", final["result"])

## 4. HITL Use Cases in Production

| Domain | Use case |
|--------|----------|
| Banking | Loan officer reviews AI recommendation before approval |
| E-commerce | Human reviews auto-generated discount campaign |
| DevOps | Engineer approves AI-suggested infrastructure change |
| Legal | Lawyer reviews AI-drafted contract clause |

## 5. Using day4 modules

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
from day4.human_in_loop import build_hitl_graph, ApprovalWorkflow
from langgraph.checkpoint.memory import MemorySaver

call_n = [0]
def mock_llm(messages):
    call_n[0] += 1
    return f"Action #{call_n[0]}: process the request"

graph    = build_hitl_graph(mock_llm, MemorySaver())
workflow = ApprovalWorkflow(graph=graph)

# Submit → pause
draft = workflow.submit("Send campaign emails to inactive users", "s1")
print("Draft:", draft)

# Approve
result = workflow.approve("s1")
print("Approved result:", result)

In [ ]:
# Reject
graph2    = build_hitl_graph(mock_llm, MemorySaver())
workflow2 = ApprovalWorkflow(graph=graph2)

draft2 = workflow2.submit("Delete all records older than 1 year", "s2")
print("Draft:", draft2)
print("Rejected result:", workflow2.reject("s2"))